In [32]:
#!pip install pytesseract pillow numpy opencv-python kraken editdistance zhipuai pdf2image transformers matplotlib

In [ ]:
import os
import numpy as np
import cv2
from PIL import Image, ImageEnhance, ImageFilter
import editdistance
from pdf2image import convert_from_path
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import subprocess
import time
import base64
import io

from kraken import pageseg
from kraken.rpred import rpred
from kraken.binarization import nlbin
from kraken.lib.models import load_any

# No pytesseract

In [34]:
# ========= PATHS =========
DATA_DIR = r"D:\GSoc 2026\Human AI - RenAIssance Test1\Data\Specefied Dataset\Test sources\Handwriting"
OUTPUT_DIR = r"D:\GSoc 2026\Human AI - RenAIssance Test1\Data\pipeline_output\PIPE_2_FIXED"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========= SETTINGS =========
USE_GLM = True
LANG = "spa"
MAX_PAGES_PER_DOC = 2

# ========= KRAKEN MODEL =========
KRAKEN_MODEL_PATH = r"D:\GSoc 2026\Human AI - RenAIssance Test1\Data\finetuned_model_12.mlmodel"

In [35]:
print("Loading Kraken model...")
try:
    kraken_model = load_any(KRAKEN_MODEL_PATH)
    print("✓ Kraken model loaded.")
except Exception as e:
    print(f"✗ Failed to load Kraken model: {e}")
    kraken_model = None

Loading Kraken model...
✓ Kraken model loaded.


In [ ]:
if USE_GLM:
    from zhipuai import ZhipuAI
    client = ZhipuAI(api_key="ZHIPU-API-KEY")

In [37]:
def load_document(path):
    if path.lower().endswith(".pdf"):
        images = convert_from_path(path, poppler_path=r'C:\\poppler\\poppler-24.08.0\\Library\\bin')
        return images
    else:
        return [Image.open(path).convert("RGB")]

In [38]:
def enhance_line_for_ocr(image):
    """
    Gentle enhancement for OCR: denoise and minimal contrast only.
    Preserves original color mode.
    """
    original_mode = image.mode
    
    if original_mode != 'L':
        gray = image.convert('L')
    else:
        gray = image
    
    # Denoise with median filter
    denoised = gray.filter(ImageFilter.MedianFilter(3))
    
    # Minimal contrast enhancement
    enhanced = ImageEnhance.Contrast(denoised).enhance(1.2)
    
    # Convert back if needed
    if original_mode == 'RGB':
        enhanced = enhanced.convert('RGB')
    
    return enhanced

In [39]:
def preprocess_for_kraken(image):
    """Preprocess specifically for Kraken (requires bi-level images)."""
    if image.mode != 'L':
        gray = image.convert('L')
    else:
        gray = image
    
    img_array = np.array(gray)
    _, binary = cv2.threshold(img_array, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    return Image.fromarray(binary)

In [ ]:
def kraken_ocr_line(line_image):
    """Run Kraken OCR on a single line image."""
    if kraken_model is None:
        return ""
    
    try:
        # Preprocess for Kraken
        if line_image.mode != 'L':
            gray = line_image.convert('L')
        else:
            gray = line_image
        
        img_array = np.array(gray)
        _, binary = cv2.threshold(img_array, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        binary_img = Image.fromarray(binary)
        
        seg = pageseg.segment(binary_img)
        
        if not seg or len(seg.lines) == 0:
            return ""
        
        predictions = rpred(kraken_model, binary_img, seg)
        text_lines = [pred.text for pred in predictions if pred.text is not None]
        return " ".join(text_lines).strip() if text_lines else ""
        
    except Exception as e:
        print(f"    Kraken line OCR failed: {e}")
        return ""


def glm_vision_line(line_image, context=""):
    """Run GLM Vision on a single line image."""
    buffered = io.BytesIO()
    if line_image.mode != 'RGB':
        rgb_image = line_image.convert('RGB')
    else:
        rgb_image = line_image
    rgb_image.save(buffered, format="PNG")
    img_base64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
    
    context_block = f"Previous line context: {context}\n" if context else ""
    
    prompt = f"""You are an expert in early modern handwritten Spanish documents.

{context_block}
Analyze the handwritten text in this image and transcribe it accurately.

RULES:
- Preserve historical/archaic spelling
- Pay attention to descenders (g, y, p, q) and ascenders (l, t, h)
- Output ONLY the transcribed text, nothing else
- If uncertain, transcribe what you see and mark with [?]

Transcription:"""

    try:
        response = client.chat.completions.create(
            model="glm-4v",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_base64}"}}
                    ]
                }
            ],
            temperature=0.1,
            max_tokens=200
        )
        time.sleep(0.5)
        return response.choices[0].message.content.strip()
        
    except Exception as e:
        print(f"    GLM Vision failed: {e}")
        return ""


def ensemble_merge_line(kraken_text, glm_text, context=""):
    """Merge Kraken and GLM outputs."""
    if not kraken_text and not glm_text:
        return ""
    if not kraken_text:
        return glm_text
    if not glm_text:
        return kraken_text
    
    context_block = f"Context (previous line): {context}\n" if context else ""
    
    prompt = f"""Merge these two OCR outputs from a historical handwritten Spanish document:

Kraken: {kraken_text}
GLM Vision: {glm_text}

{context_block}
Rules:
1. Choose the better reading for each word
2. Fix obvious OCR errors
3. Preserve historical spelling
4. Output ONLY the merged transcription

Merged:"""

    try:
        response = client.chat.completions.create(
            model="glm-4",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        time.sleep(0.5)
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"    Merge failed: {e}")
        return glm_text


def ocr_ensemble_line(line_image, prev_line=""):
    """Full ensemble OCR for a single line."""
    print("    Running Kraken OCR...")
    kraken_text = kraken_ocr_line(line_image)
    if kraken_text:
        print(f"      Kraken: {kraken_text[:80]}")
    
    print("    Running GLM Vision...")
    glm_text = glm_vision_line(line_image, prev_line)
    if glm_text:
        print(f"      GLM: {glm_text[:80]}")
    
    print("    Merging outputs...")
    merged = ensemble_merge_line(kraken_text, glm_text, prev_line)
    
    return merged

In [41]:
def cer(gt, pred):
    return editdistance.eval(gt, pred) / max(1, len(gt))

In [42]:
def glm_refine_line(line_text, context=""):
    """GLM correction for a single line."""
    context_block = f"Context (previous line): {context}\n" if context else ""
    
    prompt = f"""You are correcting OCR output from a historical handwritten Spanish document.

RULES:
- Preserve historical/archaic spelling
- Only fix obvious OCR errors (e.g., '1' for 'l', misread characters)
- Do NOT invent new words
- If unsure, keep original text

{context_block}
OCR text:
{line_text}

Corrected text (output only the corrected line, nothing else):"""

    try:
        response = client.chat.completions.create(
            model="glm-4",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        time.sleep(0.5)
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"    GLM refinement failed: {e}")
        return line_text


def glm_refine_paragraph(paragraph_text):
    """GLM paragraph-level refinement (second pass)."""
    prompt = f"""You are correcting OCR output from a historical handwritten Spanish document.

Below is a draft transcription of one page, produced line by line by OCR.
It may contain:
- OCR noise and errors at line boundaries
- Inconsistent spelling of the same name/place
- Words split across lines
- Archaic spelling

Raw transcription:
{paragraph_text}

Tasks:
1. Fix obvious OCR errors using cross-line context
2. Ensure names and places are spelled consistently
3. Reconnect words split across line boundaries
4. Preserve ALL original historical spelling — do not modernize
5. Keep the line structure

Return the corrected transcription only, preserving line breaks:"""

    try:
        response = client.chat.completions.create(
            model="glm-4",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        time.sleep(0.5)
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"    GLM paragraph refinement failed: {e}")
        return paragraph_text

In [ ]:
def deskew_line(image):
    """Gentle deskew for slightly skewed lines."""
    if image.mode != 'L':
        gray = np.array(image.convert('L'))
    else:
        gray = np.array(image)
    
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(binary > 0))
    
    if len(coords) < 20:
        return image, 0
    
    rect = cv2.minAreaRect(coords)
    angle = rect[2]
    
    if angle < -45:
        angle = 90 + angle
    
    if abs(angle) < 0.5:
        return image, 0
    
    (h, w) = gray.shape
    center = (w // 2, h // 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    
    rotated = cv2.warpAffine(
        np.array(image.convert('RGB')),
        rotation_matrix,
        (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(255, 255, 255)
    )
    
    return Image.fromarray(rotated), angle


def enhance_line(image):
    """Gentle enhancement: minimal contrast only."""
    original_mode = image.mode
    
    if original_mode != 'L':
        gray = image.convert('L')
    else:
        gray = image
    
    denoised = gray.filter(ImageFilter.MedianFilter(3))
    enhanced = ImageEnhance.Contrast(denoised).enhance(1.2)
    
    if original_mode == 'RGB':
        enhanced = enhanced.convert('RGB')
    
    return enhanced


def segment_lines_projection(image):
    """
    Line segmentation using horizontal projection with peak detection.
    This version previously detected 26-31 lines correctly.
    """
    # Convert to grayscale
    if image.mode != 'L':
        gray = np.array(image.convert('L'))
    else:
        gray = np.array(image)
    
    # Denoise
    denoised = cv2.medianBlur(gray, 3)
    
    # Binarize
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Calculate projection
    projection = np.sum(binary, axis=1)
    
    # Smooth with kernel size 21
    kernel_size = 21
    kernel = np.ones(kernel_size) / kernel_size
    projection_smooth = np.convolve(projection, kernel, mode='same')
    
    # Find peaks
    mean_proj = np.mean(projection_smooth)
    peaks, _ = find_peaks(
        projection_smooth,
        height=mean_proj * 0.3,
        distance=25,
        prominence=mean_proj * 0.2
    )
    
    print(f"    Found {len(peaks)} peaks")
    
    if len(peaks) == 0:
        print("    No peaks found, using fallback")
        return []
    
    # Group peaks into lines
    line_groups = []
    current_group = []
    max_gap = 15
    
    for peak in peaks:
        if not current_group:
            current_group.append(peak)
        elif peak - current_group[-1] <= max_gap:
            current_group.append(peak)
        else:
            line_groups.append(current_group)
            current_group = [peak]
    
    if current_group:
        line_groups.append(current_group)
    
    print(f"    Grouped into {len(line_groups)} line groups")
    
    # Extract line boundaries from each group
    lines = []
    for group in line_groups:
        y_center = int(np.mean(group))
        y_min = max(0, y_center - 30)
        y_max = min(gray.shape[0], y_center + 30)
        
        region_binary = binary[y_min:y_max, :]
        if region_binary.shape[0] == 0:
            continue
            
        row_density = np.sum(region_binary, axis=1)
        if len(row_density) == 0:
            continue
            
        text_rows = row_density > (row_density.mean() * 0.2)
        
        if np.any(text_rows):
            text_indices = np.where(text_rows)[0]
            y0 = y_min + text_indices[0] - 5
            y1 = y_min + text_indices[-1] + 5
            y0 = max(0, y0)
            y1 = min(gray.shape[0], y1)
            lines.append((y0, y1))
        else:
            y0 = max(0, y_center - 25)
            y1 = min(gray.shape[0], y_center + 25)
            lines.append((y0, y1))
    
    print(f"    Extracted {len(lines)} line boundaries")
    if lines:
        print(f"    First 3 boundaries: {lines[:3]}")
    
    # Merge overlapping or close lines
    merged = []
    merge_gap = 5
    for start, end in sorted(lines):
        if merged and start - merged[-1][1] < merge_gap:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    
    print(f"    After merge: {len(merged)} boundaries")
    if merged:
        print(f"    First 3 merged: {merged[:3]}")
    
    # Crop each line
    line_images = []
    h, w = binary.shape
    
    for idx, (y0, y1) in enumerate(merged):
        if y1 - y0 < 30:
            print(f"    Line {idx}: skipping - height {y1-y0} < 30")
            continue
        
        # Get horizontal bounds
        region_binary = binary[y0:y1, :]
        if region_binary.shape[0] == 0:
            print(f"    Line {idx}: skipping - empty region")
            continue
            
        text_cols = np.sum(region_binary, axis=0) > 0
        
        if np.any(text_cols):
            col_indices = np.where(text_cols)[0]
            line_width = col_indices[-1] - col_indices[0]
            padding = max(8, min(25, int(line_width * 0.05)))
            x0 = max(0, col_indices[0] - padding)
            x1 = min(w, col_indices[-1] + padding)
        else:
            x0, x1 = 0, w
        
        crop = image.crop((x0, y0, x1, y1))
        
        if crop.width > 80 and crop.height > 30:
            line_images.append(crop)
            if idx < 3:
                print(f"    Line {idx}: KEPT - size {crop.size}, y={y0}-{y1}")
        else:
            print(f"    Line {idx}: SKIPPED - crop too small {crop.size}, y={y0}-{y1}")
    
    print(f"    Final: {len(line_images)} lines extracted")
    return line_images

In [ ]:
def process_page(image):
    """
    Process a single page:
    1. Segment into lines
    2. Run OCR ensemble (Kraken + GLM Vision + Merge) on each line
    3. Apply paragraph-level refinement
    
    Returns (raw_ocr_text, refined_text)
    """
    # Resize image if too large
    max_dim = 2000
    w, h = image.size
    if max(w, h) > max_dim:
        ratio = max_dim / max(w, h)
        new_w = int(w * ratio)
        new_h = int(h * ratio)
        image = image.resize((new_w, new_h), Image.LANCZOS)
        print(f"  Resized image to {new_w}x{new_h}")
    
    print("  Segmenting lines...")
    line_images = segment_lines_projection(image)
    
    if not line_images:
        print(" No lines detected, skipping page")
        return "", ""
    
    print(f"  Processing {len(line_images)} lines with ensemble OCR...")
    ocr_lines = []
    prev_line = ""
    
    for idx, line_img in enumerate(line_images):
        # Skip very small lines
        if line_img.width < 80 or line_img.height < 20:
            continue
        
        # Apply gentle enhancement
        enhanced = enhance_line(line_img)
        deskewed, angle = deskew_line(enhanced)
        
        if idx % 10 == 0 and idx > 0:
            print(f"    Processed {idx}/{len(line_images)} lines")
        
        text = ocr_ensemble_line(deskewed, prev_line)
        if text and len(text) > 3:
            ocr_lines.append(text)
            prev_line = text
            print(f"    Line {idx}: {text[:80]}...")
    
    raw_text = "\n".join(ocr_lines)
    
    # Debug: print first few lines
    if ocr_lines:
        print(f"\n  First 5 lines of OCR:")
        for i, line in enumerate(ocr_lines[:5]):
            print(f"    Line {i+1}: {line[:100]}")
    else:
        print("  No text extracted")
    
    # Apply paragraph refinement
    if USE_GLM and raw_text.strip() and len(raw_text) > 100:
        print("\n  Applying paragraph-level refinement...")
        refined_text = glm_refine_paragraph(raw_text)
    else:
        refined_text = raw_text
    
    return raw_text, refined_text

In [ ]:
def process_document(path, max_pages=None):
    """
    Process a full document (PDF or image).
    
    Args:
        path: Path to PDF or image file
        max_pages: Maximum number of pages to process (None = all)
    
    Returns:
        dict with results
    """
    print("="*60)
    print(f"Processing: {os.path.basename(path)}")
    print("="*60)

    images = load_document(path)
    print(f"Loaded {len(images)} pages")

    if max_pages and max_pages < len(images):
        images = images[:max_pages]
        print(f"Processing first {max_pages} pages only")

    results = []

    for i, image in enumerate(images):
        print(f"\n--- Page {i+1}/{len(images)} ---")

        preprocessed = enhance_line_for_ocr(image) 
        raw_text, refined_text = process_page(preprocessed)

        save_path = os.path.join(
            OUTPUT_DIR,
            f"{Path(path).stem}_page{i+1}.txt"
        )

        with open(save_path, "w", encoding="utf-8") as f:
            f.write("=== RAW OCR ===\n")
            f.write(raw_text)
            f.write("\n\n=== REFINED OCR ===\n")
            f.write(refined_text)

        print(f"  Saved: {save_path}")

        results.append({
            'page': i + 1,
            'raw': raw_text,
            'refined': refined_text
        })

    return {
        'document': Path(path).stem,
        'pages': results
    }

In [46]:
# Get all files in data directory
files = [
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.lower().endswith((".pdf", ".png", ".jpg"))
]

print(f"Found {len(files)} files to process")
print("-" * 40)

all_results = []

for file in files:
    try:
        result = process_document(file, max_pages=MAX_PAGES_PER_DOC)
        all_results.append(result)
    except Exception as e:
        print(f"Error processing {file}: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*60)
print("All documents processed!")
print("="*60)
for r in all_results:
    print(f"  {r['document']}: {len(r['pages'])} pages processed")

Found 5 files to process
----------------------------------------
Processing: AHPG-GPAH 1&#x3a;1716,A.35 – 1744.pdf


d:\GSoc 2026\Human AI - RenAIssance Test1\Data\IEHHR Dataset\kraken_env_311\Lib\site-packages\PIL\Image.py:3432: DecompressionBombWarning: Image size (94080000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Loaded 3 pages
Processing first 2 pages only

--- Page 1/2 ---
  Resized image to 1500x2000
  Segmenting lines...
    Found 26 peaks
    Grouped into 26 line groups
    Extracted 26 line boundaries
    First 3 boundaries: [(np.int64(2), np.int64(70)), (np.int64(96), np.int64(165)), (np.int64(171), np.int64(240))]
    After merge: 19 boundaries
    First 3 merged: [(np.int64(2), np.int64(70)), (np.int64(96), np.int64(165)), (np.int64(171), np.int64(240))]
    Line 0: KEPT - size (1417, 68), y=2-70
    Line 1: KEPT - size (1470, 69), y=96-165
    Line 2: KEPT - size (1500, 69), y=171-240
    Final: 19 lines extracted
  Processing 19 lines with ensemble OCR...
    Running Kraken OCR...


Recognizers with segmentation types {'baselines'} will be applied to segmentation of type bbox. This will likely result in severely degraded performace


    Running GLM Vision...
      GLM: 2000 x 8 = 16000
4975 + 1744 = 6729
35.60 x 24 = 8512
    Merging outputs...
    Line 0: 2000 x 8 = 16000
4975 + 1744 = 6729
35.60 x 24 = 8512...
    Running Kraken OCR...


...

    Running GLM Vision...
      GLM: Li muy los habían comido, y los orgullosos repartían conmigo al uno por libra e 
    Merging outputs...
    Line 5: Li muy los habían comido, y los orgullosos repartían conmigo al uno por libra e ...
    Running Kraken OCR...


Recognizers with segmentation types {'baselines'} will be applied to segmentation of type bbox. This will likely result in severely degraded performace


    Running GLM Vision...
      GLM: ni declaracion una verdad total ni parcialmente, y ado hacian aplicar que si mas
    Merging outputs...
    Line 6: ni declaracion una verdad total ni parcialmente, y ado hacian aplicar que si mas...
    Running Kraken OCR...


Recognizers with segmentation types {'baselines'} will be applied to segmentation of type bbox. This will likely result in severely degraded performace


    Running GLM Vision...
      GLM: ni declaracion una verdad total ni parcialmente, y ado hacian aplicar que si más
    Merging outputs...
    Line 7: ni declaracion una verdad total ni parcialmente, y ado hacian aplicar que si más...
    Running Kraken OCR...
    Running GLM Vision...
      GLM: ni declaracion una verdad total ni parcialmente, y ado hacian aplicar que si más
    Merging outputs...
    Line 8: ni declaracion una verdad total ni parcialmente, y ado hacian aplicar que si más...

  First 5 lines of OCR:
    Line 1: Había que comer mucho ahora aproximadamente que allí pasará que no hay
    Line 2: de su libro llevando aprendizaje, morgan que se comprendería donde bueno para el hombre de las refac
    Line 3: Había una vez un libro llevando aprendizaje, morgan que se comprendería donde bueno para el hombre d
    Line 4: Li muy los habían comido, y los orgullosos repartían conmigo al uno por libra e indemnizaban cada un
    Line 5: Li muy los habían comido, y los orgulloso